# 04 — The business case: what the model is for, and how we would know it worked

The competition question was *how well can churn be ranked* — answered in
`notebooks/99_final_writeup.ipynb` at OOF AUC **0.916391**. This notebook asks the
question a business asks instead: **what decision does that ranking support, what
is it worth, and what would we have to measure to find out whether it worked?**

**Nothing here is trained.** Every score comes from `experiments/oof_1024_141318_lgbm.npy`
— the saved out-of-fold predictions of the bundle the API actually serves, which
are bitwise identical to champion run `0904_202729_lgbm` (asserted in cell 2). OOF is
the right instrument: each row was scored by a model that never saw it, which is
the closest offline analogue of production.

The money in §4–§6 rests on **four stated assumptions**, collected in one cell so
a reader can replace them. Everything before §4 is measurement only.

`docs/use-case.md` is the business-facing write-up of what follows.

In [ ]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

from src import config as C
from src.utils import savefig

SERVED = "1024_141318_lgbm"  # the bundle in models/, and what the API serves
CHAMP = "0904_202729_lgbm"  # the run that scored public LB 0.91393

# only two raw columns are needed: the label and the revenue it puts at risk
raw = pd.read_csv(C.TRAIN_FILE, usecols=["Churn", "MonthlyCharges"])
y = (raw.Churn == "Yes").astype("int8").to_numpy()
mrr = raw.MonthlyCharges.to_numpy(dtype=float)  # monthly recurring revenue, GBP

p = np.load(C.EXPERIMENTS / f"oof_{SERVED}.npy")
assert np.array_equal(p, np.load(C.EXPERIMENTS / f"oof_{CHAMP}.npy")), (
    "served bundle's OOF differs from the champion's — do not build a business case on it"
)
N = len(y)
print(f"{N:,} customers   churn rate {y.mean():.4f}   served OOF == champion OOF")

## 1. The problem, sized in money

A churn rate is a headcount statistic, and headcount is not what a telco loses.
The first thing worth measuring is whether churn is **revenue-weighted** — whether
the customers leaving are worth more or less than the ones staying.

In [ ]:
facts = pd.DataFrame(
    [
        ("customers", f"{N:,}"),
        ("churn rate (headcount)", f"{y.mean():.2%}"),
        ("churners", f"{y.sum():,}"),
        ("total MRR", f"GBP {mrr.sum():,.0f} / month"),
        ("MRR carried by churners", f"GBP {mrr[y == 1].sum():,.0f} / month"),
        ("churn rate (revenue)", f"{mrr[y == 1].sum() / mrr.sum():.2%}"),
        ("ARPU, churners", f"GBP {mrr[y == 1].mean():.2f} / month"),
        ("ARPU, non-churners", f"GBP {mrr[y == 0].mean():.2f} / month"),
        ("revenue weighting", f"{mrr[y == 1].mean() / mrr[y == 0].mean():.2f}x"),
    ],
    columns=["fact", "value"],
)
print(facts.to_string(index=False))

**Read.** Churn takes **27.9% of revenue** while taking 22.5% of customers,
because a churner pays **£81.60/month against a non-churner's £61.29 — 1.33×**.
The customers walking out are the expensive ones.

Two consequences, and the whole rest of the notebook follows from them:

1. The loss is **£10.9M of monthly recurring revenue**, not "22.5% churn". That is
   the number a retention budget is set against.
2. Because value is unevenly spread, *whom you call* has two dimensions —
   how likely they are to leave **and** what leaving costs. §5 measures what the
   second one is worth.

## 2. What the ranking buys

The operational shape is a ranked call list worked from the top until the budget
runs out — which is why the model was selected on ROC-AUC, a ranking metric, and
why accuracy would have been the wrong target (predicting "nobody churns" scores
77.5% accurate and supports no decision).

So the question is not the AUC. It is: **at a given call capacity, how many
churners and how much at-risk revenue does the list reach?**

In [ ]:
by_p = np.argsort(-p)
DEPTHS = (0.01, 0.05, 0.10, 0.20, 0.30, 0.50)

rows = []
for k in DEPTHS:
    top = by_p[: int(k * N)]
    churn_mrr = mrr[top][y[top] == 1].sum()
    rows.append(
        {
            "call the top": f"{k:.0%}",
            "customers": len(top),
            "precision": y[top].mean(),
            "churners reached": y[top].sum() / y.sum(),
            "lift": y[top].mean() / y.mean(),
            "at-risk MRR reached": churn_mrr / mrr[y == 1].sum(),
        }
    )
depth_table = pd.DataFrame(rows)
print(depth_table.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

## 3. Can these probabilities be used as money?

§4 is about to multiply the model's output by a sum of money, which assumes the
output means what it says: that customers scored 0.83 churn 83% of the time. AUC
does **not** check that — it is invariant to any monotone rescaling, so a model
can rank perfectly and still be numerically wrong everywhere.

`docs/serving.md` is careful about this: the API returns "a probability, not a
decision", and no operating threshold was ever chosen. Calibration is what decides
whether a threshold *can* be chosen on economic grounds, so it has to be measured
rather than hoped for.

In [ ]:
cal = (
    pd.DataFrame({"decile": pd.qcut(p, 10, labels=False, duplicates="drop"), "p": p, "y": y})
    .groupby("decile")
    .agg(n=("y", "size"), predicted=("p", "mean"), observed=("y", "mean"))
)
cal["pred/obs"] = cal.predicted / cal.observed

brier = np.mean((p - y) ** 2)
brier_base = np.mean((y.mean() - y) ** 2)
print(
    f"mean predicted {p.mean():.5f}  vs  observed {y.mean():.5f}   ratio {p.mean() / y.mean():.4f}"
)
print(f"Brier {brier:.5f}   vs {brier_base:.5f} for a constant base-rate forecast")
print(f"skill score {1 - brier / brier_base:.4f}\n")
print(cal.to_string(float_format=lambda v: f"{v:.4f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

# (a) gains in two currencies: churners found, and at-risk revenue reached
frac = np.arange(1, N + 1) / N
axes[0].plot(frac, np.cumsum(y[by_p]) / y.sum(), lw=2, label="churners reached")
axes[0].plot(
    frac,
    np.cumsum(mrr[by_p] * y[by_p]) / mrr[y == 1].sum(),
    lw=2,
    label="at-risk MRR reached",
)
axes[0].plot([0, 1], [0, 1], "k--", lw=1, label="random calling")
axes[0].axvline(0.10, color="tab:red", ls=":", lw=1.5, label="top decile")
axes[0].set(
    xlabel="fraction of customers called",
    ylabel="fraction reached",
    title="Cumulative gains, OOF",
)
axes[0].legend(loc="lower right", fontsize=9)

# (b) calibration — the check that licenses the economics in section 4
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
axes[1].plot(cal.predicted, cal.observed, "o-", lw=2, label="served model, by decile")
axes[1].set(
    xlabel="mean predicted probability",
    ylabel="observed churn rate",
    title="Calibration (OOF, deciles)",
)
axes[1].legend(loc="upper left", fontsize=9)

fig.tight_layout()
savefig(fig, "04_value_at_risk")

**Read.** The model is **already calibrated, without any post-hoc correction**:
mean predicted 0.22517 against an observed 0.22521 (ratio 0.9999), and the top
decile — the only one a shallow campaign ever calls — predicts 0.8302 against an
observed 0.8306. The worst decile error is the bottom one, 0.0011 vs 0.0011 in
absolute terms and 7% in relative terms, where the money is negligible anyway.
Brier 0.09505 against 0.17449 for a constant forecast is a skill score of 0.455.

That is not luck: LightGBM's binary objective *is* log loss, a proper scoring
rule, and 594k rows leave little room for the fitted probabilities to drift from
observed frequencies. It does mean the output can be treated as an expected
value, which is the licence §4 needs. **If this check had failed, the right move
would be isotonic or Platt calibration fitted on OOF — not a hand-picked
threshold.**

## 4. The decision rule, and how deep to call

Contacting one customer has a cost; saving one has a value. Writing down both
turns the probability into a decision without anyone choosing a threshold by feel.

For a customer with churn probability $p$ and monthly revenue $m$, where $c$ is
the cost of one contact, $s$ the share of contacted churners actually retained,
and $H$ the months a saved customer stays:

$$\mathbb{E}[\text{contact}] = \underbrace{p \cdot s \cdot m \cdot H}_{\text{expected revenue retained}} - c$$

Contact anyone whose expected value is positive, which rearranges to a threshold
not on $p$ but on the **product** $p \cdot m$:

$$p \cdot m > \frac{c}{s \cdot H}$$

$s$ is the one quantity no observational model can supply — it is a property of
the *intervention*, not the customer, and §6 is about measuring it. Everything
below is therefore parameterised by it.

In [ ]:
# ---- ASSUMPTIONS. Not measured here; replace with your own and re-run. -----
COST_PER_CONTACT = 5.00  # GBP: outbound contact + offer admin
SAVE_RATE = 0.10  # share of contacted churners actually retained  <- the unknown
HORIZON_MONTHS = 12  # months a saved customer is assumed to stay
# ---------------------------------------------------------------------------

value = mrr * HORIZON_MONTHS  # GBP value of saving this customer
ev = p * SAVE_RATE * value - COST_PER_CONTACT  # expected value of one contact

print(f"contact if  p * m > c / (s * H) = {COST_PER_CONTACT / (SAVE_RATE * HORIZON_MONTHS):.2f}")
print(f"customers clearing the bar: {(ev > 0).sum():,} ({(ev > 0).mean():.1%} of the base)")

In [ ]:
by_ev = np.argsort(-(p * mrr))  # rank by expected value, not by probability

rows = []
for k in DEPTHS:
    top = by_ev[: int(k * N)]
    saves = (p[top] * SAVE_RATE).sum()
    gross = (p[top] * SAVE_RATE * value[top]).sum()
    cost = COST_PER_CONTACT * len(top)
    rows.append(
        {
            "depth": f"{k:.0%}",
            "contacts": len(top),
            "expected saves": saves,
            "revenue retained": gross,
            "campaign cost": cost,
            "net": gross - cost,
            "ROI": gross / cost,
            "cost per save": cost / saves,
            "EV of the last call": ev[top][-1],
        }
    )
econ = pd.DataFrame(rows)
print(econ.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

In [ ]:
CAP = int(0.10 * N)  # a top-decile campaign
top = by_ev[:CAP]
SAVE_GRID = (0.02, 0.05, 0.10, 0.15, 0.20)
gross_at = {s: (p[top] * s * value[top]).sum() for s in SAVE_GRID}
breakeven = COST_PER_CONTACT * CAP / (p[top] * value[top]).sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

# (a) where to stop: net value rises while the marginal call collapses
x = np.arange(len(econ))
axes[0].bar(x, econ["net"] / 1e6, color="tab:blue", label="net value of the campaign")
axes[0].set(xticks=x, xlabel="depth of the call list", ylabel="GBP m", ylim=(0, None))
axes[0].set_xticklabels(econ.depth)
axes[0].set_title("Net value, and the value of the last call made")
twin = axes[0].twinx()
twin.plot(x, econ["EV of the last call"], "o-", color="tab:red", lw=2, label="EV of the last call")
twin.set(ylabel="GBP per contact", ylim=(0, None))  # share the zero line with the bars
h1, l1 = axes[0].get_legend_handles_labels()
h2, l2 = twin.get_legend_handles_labels()
axes[0].legend(h1 + h2, l1 + l2, loc="center left", fontsize=9)

# (b) the only unknown is the save rate -- and the sign does not depend on it
axes[1].plot(
    [100 * s for s in SAVE_GRID],
    [(gross_at[s] - COST_PER_CONTACT * CAP) / 1e6 for s in SAVE_GRID],
    "o-",
    lw=2,
)
axes[1].axhline(0, color="k", lw=1)
axes[1].axvline(
    100 * breakeven, color="tab:red", ls=":", lw=1.5, label=f"break-even s = {breakeven:.2%}"
)
axes[1].set(
    xlabel="save rate s, % of contacted churners retained",
    ylabel="net value, GBP m",
    title=f"Sensitivity of a {CAP:,}-contact campaign to s",
)
axes[1].legend(loc="upper left", fontsize=9)

fig.tight_layout()
savefig(fig, "04_campaign_economics")

print(
    f"break-even save rate at 10% depth: {breakeven:.4%}  (1 save per {1 / breakeven:.0f} contacts)"
)

**Read — this is the answer to "how far down the list do we go?".**

Under the stated assumptions the rule puts **300,973 customers (50.7%)** above
the bar, so a *budget-unconstrained* campaign would call half the base for a net
£11.3M. That is the wrong way to read the table. Look at the last two columns
instead:

| depth | ROI | cost per save | EV of the last call |
|---|---|---|---|
| 1% | 21.7× | £54 | £99 |
| 10% | 17.8× | £62 | £68 |
| 30% | 12.5× | £83 | £22 |
| 50% | 8.6× | £115 | £0.33 |

Net value keeps rising to 50% depth while the **marginal** call falls from £99 to
£0.33 and cost per save doubles. So the depth is a **capacity** decision, not a
model one: with limited agent hours, every hour spent at 30% depth is worth a
third of an hour spent at 1%. **Go shallow and deepen the offer, not the list.**

And the campaign's *sign* does not depend on the unknown. Break-even is a
**0.56% save rate — one save per 178 contacts.** Any plausible $s$ makes a
top-decile campaign profitable; what $s$ determines is the size of the prize, not
whether to run it. That is worth stating precisely because it is the usual
objection to acting on a churn model at all.

## 5. Rank by value, not by risk

§1 found that churners are worth 1.33× non-churners, and §4's rule thresholds
$p \cdot m$ rather than $p$. Both say the call list should be ordered by expected
value. The ranking the model hands over is ordered by probability. **What does the
difference cost?**

In [ ]:
rows = []
for name, key in [("probability p", p), ("expected value p x m", p * mrr)]:
    t = np.argsort(-key)[:CAP]
    rows.append(
        {
            "ranked by": name,
            "precision": y[t].mean(),
            "churners reached": y[t].sum() / y.sum(),
            "at-risk MRR reached": mrr[t][y[t] == 1].sum(),
            "share of at-risk MRR": mrr[t][y[t] == 1].sum() / mrr[y == 1].sum(),
            "expected net value": ev[t].sum(),
        }
    )
cmp = pd.DataFrame(rows)
print(cmp.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
d_mrr = cmp["at-risk MRR reached"].iloc[1] - cmp["at-risk MRR reached"].iloc[0]
d_ev = cmp["expected net value"].iloc[1] - cmp["expected net value"].iloc[0]
print(f"\nvalue ranking vs probability ranking, same {CAP:,} contacts:")
print(f"  at-risk MRR reached  {d_mrr:+,.0f} / month")
print(f"  expected net value   {d_ev:+,.0f}")
print(f"  precision            {cmp.precision.iloc[1] - cmp.precision.iloc[0]:+.4f}")

**Read.** Same 59,419 contacts, reordered: **+£125,721/month of at-risk revenue
reached and +£149,043 of expected net value, for 1.7pp less precision.**

That is the cleanest example in this repo of a business metric and a model metric
disagreeing. The value-ranked list is *worse* at the thing the model was scored
on and *better* at the thing the campaign is for, because it trades away cheap
near-certain churners for expensive probable ones. A model-selection process that
only ever looked at AUC would never find this, and it costs nothing to apply —
`monthlycharges` is already in the request the API receives.

**The honest caveat:** $m$ is current monthly revenue, a proxy for customer
lifetime value. A real deployment would use margin rather than revenue, and a
tenure-dependent horizon rather than a flat 12 months. Both refine the same rule.

## 6. Measuring the effect — the part the model cannot do

Everything above is an *expectation* computed from OOF predictions. None of it is
evidence that calling anyone prevents churn. The model ranks who is **at risk**;
a campaign needs to know who is **persuadable**, and those are different people —
a customer who will leave whatever you do and one who can be talked round look
identical to a model fitted on observational data.

So the save rate $s$ has to be measured, and the only instrument that measures it
is a **randomised holdout**: take the ranked list, withhold contact from a random
share of it, and compare observed churn between the arms. The question is how big
a holdout buys an answer worth having.

In [ ]:
z = norm.ppf(0.975) + norm.ppf(0.80)  # two-sided 5%, 80% power
p1 = y[by_p[:CAP]].mean()  # observed churn rate in the called decile

print(f"churn rate in the called decile: {p1:.4f}\n")
rows = []
for d in (0.01, 0.02, 0.03, 0.05):
    p2 = p1 - d
    n = z**2 * (p1 * (1 - p1) + p2 * (1 - p2)) / d**2
    rows.append(
        {
            "detect (absolute)": f"{d * 100:.0f}pp",
            "implied save rate s": f"{d / p1:.1%}",
            "customers per arm": f"{n:,.0f}",
            "holdout as % of campaign": f"{n / CAP:.1%}",
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

**Read.** A **10% holdout of a top-decile campaign — 5,942 customers — resolves a
2pp absolute reduction in churn**, i.e. an $s$ of about 2.4%, at 80% power and 5%
two-sided. That is comfortably below the 10% assumed in §4 and far above the 0.56%
break-even, so this design can tell "worth running" from "not worth running".

Note what the table also says: a 1pp effect needs a **38% holdout**, which is not
a measurement a business will agree to. Effects that small are not detectable at
this campaign size — worth knowing before promising to measure them.

### The three tiers of measurement, and what is already instrumented

| tier | question | metric | where it lives today |
|---|---|---|---|
| **model** | does it rank risk? | OOF AUC on frozen folds, lift@k, calibration | `experiments.csv`, `src.log_lb`, this notebook; folds pinned at fingerprint `13feb807` and CI fails if it moves |
| **decision** | is the campaign worth running? | net value, ROI, cost per save, break-even $s$ | §4–§5 here; recompute whenever $c$, $s$ or $H$ change |
| **effect** | did contacting them help? | churn difference vs randomised holdout, realised saves, revenue retained | **not built** — needs the experiment above |
| **system** | is it still working? | `churn_predictions_total`, `churn_prediction_latency_seconds`, `churn_predicted_probability`, `churn_out_of_range_inputs_total{field}` | `/metrics` → Prometheus; `make drift` for input drift |

The first, second and fourth rows are reproducible from this repo today. The third
is a deliberate gap: it cannot be filled offline, and no amount of AUC substitutes
for it.

**The ordering matters.** A tier-1 win of +0.0002 AUC is below this setup's noise
floor (`learnings.md`: ±0.00007 paired) and changes no decision. A tier-3
measurement of $s$ moves the whole campaign's value by the ratios in §4's
sensitivity panel. **The cheapest remaining improvement is not a better model.**

## 7. What this does not license

- **The data is synthetic** — a generated expansion of the Telco table. The
  £10.9M, the 1.33× revenue weighting and every pound below it are arithmetic on
  a generator's output. The *method* transfers; the figures do not.
- **£5, 10% and 12 months are assumptions, not findings.** They are isolated in
  one cell for exactly that reason. The only one whose value changes the decision
  rather than its size is $s$, and §6 is how to get it.
- **Nothing here is causal.** "Month-to-month contracts raise risk" (notebook 99
  §5) is a statement about the data. Contract type is partly a *consequence* of
  intent to stay, so migrating a customer to an annual contract need not lower
  their risk.
- **No repeat-contact suppression is possible.** There is no `customerid` in this
  dataset (`experiments.md` → Validation decision), so a real deployment would
  need contact history the model has never seen — someone called three times this
  quarter should not be called again because their score is still high.
- **A probability is not a decision**, and this notebook has not changed that.
  `src/api.py` still returns `churn_probability` plus a `risk_band`; the threshold
  derived in §4 belongs to the campaign that spends the budget, not to the
  service. That separation is deliberate — see `docs/serving.md`.